In [1]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
from typing import List, Tuple

In [2]:
output_path = "f0_OSMOSYS_ECU_Output.csv"
df = pd.read_csv(output_path)
ndc = pd.read_csv("docs/NDCOutput.csv")
planmicc = pd.read_csv("docs/PLANMICCOutput.csv")

C:\Users\adria\AppData\Local\Temp\ipykernel_20792\2844354578.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(output_path)
C:\Users\adria\AppData\Local\Temp\ipykernel_20792\2844354578.py:4: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  planmicc = pd.read_csv("docs/PLANMICCOutput.csv")


In [3]:
initial_year = 2010
final_year = 2035

In [4]:
# Filter only co2e emissions 
def get_model_emissions(dataFrame: pd.DataFrame, emissions:List[str]) -> pd.DataFrame:
    emission_filters = (dataFrame['Year'] <= final_year) & (~dataFrame['Emission'].isna()) & (~dataFrame['AnnualEmissions'].isna()) & (dataFrame['Emission'].isin(emissions))
    emissions_output = dataFrame.loc[emission_filters, ['Strategy', 'Emission','Year', 'AnnualEmissions']]
    missing_emissions =  set(emissions) - set(emissions_output['Emission'].unique())
    if len(missing_emissions) > 0:
        print("WARNING: The following emissions are missing:")
        print(missing_emissions)
    emissions_output['AnnualEmissions'] = emissions_output['AnnualEmissions'] * 1000
    waste_emissions = emissions_output.pivot_table(index = ['Year'], columns='Strategy' ,values = 'AnnualEmissions', aggfunc='sum').reset_index()
    return waste_emissions

In [5]:
def draw_model_emission_scatter(name: str, dataFrame: pd.DataFrame) -> List[go.Scatter]:
    data = []
    for scenario in dataFrame.columns[1:]:
        data.append(
            go.Scatter(
                name=f"{scenario}({name})", x=dataFrame["Year"], y=dataFrame[scenario]
            )
        )

    return data


# Fig. Emisiones Totales
def draw_emissions_trayectory(emissions: List[Tuple[str, pd.DataFrame]], title: str):
    fig_data = []
    for model in emissions:
        fig_data += draw_model_emission_scatter(model[0], model[1])

    fig = go.Figure(data=fig_data)

    fig.update_yaxes(
        mirror=True,
        showline=False,
        gridcolor="lightgrey",
    )

    fig.update_layout(
        plot_bgcolor="white",
        title={
            "text": title,
        },
        title_x=0.5,
        yaxis_title="[GgCO2e]",
    )

    fig.update_traces(
        hovertemplate="""
    <span>Año:</span> <b>%{x}</b>
    <span>Emisiones:</span> <b>%{y:.3f}</b>
    """
    )

    fig.show()

In [6]:
ndc_emissions = get_model_emissions(ndc, [
    #'CH4_WASTE', 
    'CO2e_WASTE', 
    #'N2O_WASTE', 
    #'CO2_WASTE'
])
planmicc_emissions= get_model_emissions(planmicc, [
    'CO2e', 
    #'FERT_ORG', 
    #'RM', 
    #'SALUD', 
    #'HWT'
])
df_emissions = get_model_emissions(df, [
    #'CH4_WASTE', 
    'CO2e_WASTE', 
    #'FERT_ORG', 
    #'N2O_WASTE', 
    #'CO2_WASTE', 
    #'RM', 
    #'SALUD', 
    #'HWT'    
])
draw_emissions_trayectory([
    ('Current', df_emissions), 
    ('PLANMICC', planmicc_emissions), 
    ('NDC', ndc_emissions)
],
title="Trayectoria Emisiones sector Residuos"
)

In [7]:
def get_grouped_techs_emissions_by_column(dataFrame: pd.DataFrame,column:str, strategy:str, emission_label:str, groupings:dict):
  fuel_groups = {x: k for k, v in groupings.items() for x in v}
  fuel_waste_filters =  (dataFrame['Year'] <= final_year) & (~dataFrame['Technology'].isna()) & (~dataFrame[column].isna()) & (dataFrame['Strategy'] == strategy)
  
  if emission_label != '' and emission_label != None:
    fuel_waste_filters = fuel_waste_filters & (dataFrame['Emission'] == emission_label)

  fuel_waste_output = dataFrame.loc[fuel_waste_filters, ['Technology','Year', column]]
  fuel_waste_output['Technology'] = fuel_waste_output['Technology'].map(fuel_groups) # replace tech codes for group codes, ignore if not mapped
  fuel_waste = fuel_waste_output.pivot_table(index = ['Year'], columns='Technology' ,values = column, aggfunc='sum').reset_index()
  return fuel_waste


def draw_tech_waste_by_column(dataFrame: pd.DataFrame, strategy:str, title:str, y_label:str):
    # Fig. Emissions per Technology vs Year
    fig = px.area(dataFrame, 
                x="Year", 
                y=dataFrame.columns[1:], 
                labels={
                    "variable":"",
                    "Year": "Año",
                    "value": y_label
                },
                )

    fig.update_yaxes(
        mirror=True,
        showline=False,
        gridcolor='lightgrey',
    )

    fig.update_layout(
        plot_bgcolor="white",
        title = {
            "text":f"{title} - {strategy}",
        },
        title_x=0.5,
    )

    fig.update_layout(legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.3,     
            xanchor="right",
            x=1
    ))

    fig.update_traces(hovertemplate="""
    <span>Año:</span> <b>%{x}</b>
    <span>Valor:</span> <b>%{y:.3f}</b>
    """)
    fig.show()


In [8]:
# Scenario

# current
current_scenarios = list(df['Strategy'].unique())

# planmicc
planmicc_scenarios = list(planmicc['Strategy'].unique())

# ndc
ndc_scenarios = list(ndc['Strategy'].unique())

In [9]:
# Current Emissions Disaggregated Emissions
emissions_techs_groups = {
  "Tratamiento biologico de residuos sólidos": ["COMPOST"],
  "Relleno sanitario": ["LANDFILL"],
  "Botadero a cielo abierto": ["NO_CONTR_OD"],
  "Inicineracion": ["INCIN"],
  "Aguas Industriales": ["IWW"],
  "Aguas Domésticas": ["TWWWOT", "SEWERWW", "STWW"],
}

# current

for scenario in current_scenarios:
    waste_emission = get_grouped_techs_emissions_by_column(df,'AnnualTechnologyEmission', scenario, 'CO2e_WASTE', emissions_techs_groups)
    draw_tech_waste_by_column( waste_emission, scenario ,"Emisiones sectoriales desagregadas (Current)", "[Mt CO2eq]")

# planmicc

# for scenario in planmicc_scenarios:
#     waste_emission = get_grouped_techs_emissions_by_column(planmicc, 'AnnualTechnologyEmission', scenario, 'CO2e', emissions_techs_groups)
#     draw_tech_waste_by_column( waste_emission, scenario ,"Emisiones sectoriales desagregadas (PLANMICC)", "[Mt CO2eq]")


# ndc

# for scenario in ndc_scenarios:
#     waste_emission = get_grouped_techs_emissions_by_column(ndc, 'AnnualTechnologyEmission', scenario, 'CO2e_WASTE', emissions_techs_groups)
#     draw_tech_waste_by_column( waste_emission, scenario ,"Emisiones sectoriales desagregadas (NDC)", "[Mt CO2eq]")

In [10]:
def get_fuel_use_production_by_strategy(dataFrame: pd.DataFrame, fuel:str, strategy:str):
    
    fuel_waste_filters = (dataFrame['Year'] <= final_year) & (dataFrame['Fuel']==fuel) & (dataFrame['Strategy'] == strategy)
    fuel_waste_outputs = dataFrame.loc[fuel_waste_filters, ['Year', 'UseByTechnology', 'ProductionByTechnology']]
    fuel_waste = fuel_waste_outputs.groupby(['Year']).agg('sum').reset_index()
    return fuel_waste

# Solid Waste - CO2eq emissions
def draw_fuel_use_production_by_strategy(fuel_waste: pd.DataFrame, title:str):
    fig = px.line(fuel_waste, 
                x="Year", 
                y=fuel_waste.columns[1:], 
                labels={
                    "variable":"",
                    "Year": "",
                    "value": "[MtCO2eq]",
                },
                title=title)

    fig.update_yaxes(
        mirror=True,
        showline=False,
        gridcolor='lightgrey',
    )

    fig.update_layout(
        plot_bgcolor="white",
        title = {
            "text":title,
        },
        title_x=0.5,
    )

    fig.update_traces(hovertemplate="""
    <span>Año:</span> <b>%{x}</b>
    <span>Valor:</span> <b>%{y:.3f}</b>
    """)

    fig.show()


In [11]:
# Total Solid Waste Use and Production

# Current
for scenario in current_scenarios:
    fuel_waste = get_fuel_use_production_by_strategy(df, 'TSW', scenario)
    draw_fuel_use_production_by_strategy(fuel_waste, f"Uso y Producción de Residuos Sólidos Totales(Current) - {scenario}")


# # planmicc
# for scenario in planmicc_scenarios:
#     fuel_waste = get_fuel_use_production_by_strategy(planmicc, 'TSW', scenario)
#     draw_fuel_use_production_by_strategy(fuel_waste, f"Uso y Producción de Residuos Sólidos Totales(PLANMICC) - {scenario}")


# # ndc
# for scenario in ndc_scenarios:
#     fuel_waste = get_fuel_use_production_by_strategy(ndc, 'TSW', scenario)
#     draw_fuel_use_production_by_strategy(fuel_waste, f"Uso y Producción de Residuos Sólidos Totales(NDC) - {scenario}")

In [12]:

# Total Water Waste Use and Production

# Current
for scenario in current_scenarios:
    fuel_waste = get_fuel_use_production_by_strategy(df, 'TWW', scenario)
    draw_fuel_use_production_by_strategy(fuel_waste, f"Uso y Producción de Aguas Residuales Totales(Current) - {scenario}")



In [13]:

grouped_techs = {
  "Aguas residuales con tratamiento": ["SEWERWW"],
  "Aguas residuales sin tratamiento": ["TWWWOT"],
  #"Aguas residuales domesticas nitrógeno": ["DWW_N"],
  "Aguas residuales totales": ["T5TWWTWW"],
}

# current
for scenario in current_scenarios:
    waste_emission = get_grouped_techs_emissions_by_column(df,'ProductionByTechnology', scenario,'', grouped_techs)
    draw_tech_waste_by_column( waste_emission, scenario ,"Generación de aguas residuales (Current)", "[Mt]")

# # planmicc
# for scenario in planmicc_scenarios:
#     waste_emission = get_grouped_techs_emissions_by_column(planmicc,'ProductionByTechnology', scenario,'', grouped_techs)
#     draw_tech_waste_by_column( waste_emission, scenario ,"Generación de aguas residuales (PLANMICC)", "[Mt]")


In [14]:

grouped_techs = {
  "Compostaje": ["COMPOST"],
  "Relleno sanitario": ["LANDFILL"],
  "Botadero a cielo abierto": ["NO_CONTR_OD"],
}

# current
for scenario in current_scenarios:
    waste_emission = get_grouped_techs_emissions_by_column(df,'ProductionByTechnology', scenario,'', grouped_techs)
    draw_tech_waste_by_column( waste_emission, scenario ,"Disposición final de residuos sólidos (Current)", "[Mt]")

# # planmicc
# for scenario in planmicc_scenarios:
#     waste_emission = get_grouped_techs_emissions_by_column(planmicc,'ProductionByTechnology', scenario,'', grouped_techs)
#     draw_tech_waste_by_column( waste_emission, scenario ,"Disposición final de residuos sólidos (PLANMICC)", "[Mt]")

In [15]:
grouped_techs = {
  "Separación de residuos inorgánicos en sitio de disposición final": ["OSS_INORG"],
  "Separación de residuos orgánicos en sitio de disposición final": ["OSS_ORG"],
  "No separación de residuos mezclados en sitio de disposición final": ["NO_OSS_BLEND"],
}

# current
for scenario in current_scenarios:
    waste_emission = get_grouped_techs_emissions_by_column(df,'ProductionByTechnology', scenario,'', grouped_techs)
    draw_tech_waste_by_column( waste_emission, scenario ,"Separación en el sitio de disposición final (Current)", "[Mt]")

# # planmicc
# for scenario in planmicc_scenarios:
#     waste_emission = get_grouped_techs_emissions_by_column(planmicc,'ProductionByTechnology', scenario,'', grouped_techs)
#     draw_tech_waste_by_column( waste_emission, scenario ,"Separación en el sitio de disposición final (PLANMICC)", "[Mt]")

In [16]:
grouped_techs = {
  "Recolección diferenciada de residuos inorgánicos": ["INORG_DCOLL"],
  "Recolección diferenciada de residuos orgánicos": ["ORG_DCOLL"],
  "Recolección no diferenciada de residuos mezclados": ["BLEND_NO_DCOLL"],
}

# current
for scenario in current_scenarios:
    waste_emission = get_grouped_techs_emissions_by_column(df,'ProductionByTechnology', scenario,'', grouped_techs)
    draw_tech_waste_by_column( waste_emission, scenario ,"Recolección (Current)", "[Mt]")

# # planmicc
# for scenario in planmicc_scenarios:
#     waste_emission = get_grouped_techs_emissions_by_column(planmicc,'ProductionByTechnology', scenario,'', grouped_techs)
#     draw_tech_waste_by_column( waste_emission, scenario ,"Recolección (PLANMICC)", "[Mt]")